# ⚡ Challenge 20: 대용량 처리 튜닝 — 프로덕션 설정 탐구

**난이도:** ⭐⭐⭐⭐⭐  
**실행 요구사항:** `docker compose up -d`

---

## 목표

> 기본 설정과 튜닝된 설정의 **실측 성능 차이를 직접 확인**합니다.  
> 프로덕션에서 메시지 브로커를 최대한 활용하는 방법을 학습합니다.

### 실험할 튜닝 포인트

| 브로커 | 파라미터 | 기본값 | 튜닝값 | 예상 효과 |
|--------|---------|--------|--------|----------|
| **Kafka** | `batch_size` | 16,384 B | 65,536 B | 처리량 ↑ |
| **Kafka** | `linger_ms` | 0 ms | 10 ms | 처리량 ↑ (레이턴시 소폭 ↑) |
| **Kafka** | `compression_type` | none | lz4 | 네트워크 부하 ↓ |
| **Redis** | 단건 SET vs Pipeline | 1개씩 | 배치 | 처리량 4~10x |
| **RabbitMQ** | confirm_delivery | True | False | 처리량 ↑ (내구성 ↓) |

In [ ]:
# 환경 설정
import asyncio
import json
import time
import statistics
import httpx
import redis.asyncio as aioredis

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    PLOT_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
except ImportError:
    PLOT_AVAILABLE = False
    print("⚠️  matplotlib 없음 — 수치 결과만 출력됩니다")

BASE_URL = "http://localhost:8000"

async def api(method: str, path: str, **kwargs):
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=120.0) as c:
        return (await getattr(c, method)(path, **kwargs)).json()

# 헬스체크
health = await api("get", "/health")
print("✅ 브로커 연결 상태:")
for k, v in health.items():
    print(f"  {'🟢' if v in ('running','connected') else '🔴'} {k}: {v}")

---

## 실험 1: Redis — 단건 vs Pipeline 처리량 비교

### 왜 Pipeline이 빠른가?

```text
[단건 방식]                    [Pipeline 방식]

Client → SET key1 → Server    Client → SET key1
Client ← OK       ← Server            SET key2   → Server (한 번에)
Client → SET key2 → Server            SET key3
Client ← OK       ← Server    Client ←       OK,OK,OK ← Server
Client → SET key3 → Server
Client ← OK       ← Server

RTT: N번 (네트워크 왕복)        RTT: 1번 (한 번에 전송)
```

**왜 beanllm이 Redis Pipeline을 사용하는가?**  
대량 문서 임베딩 저장, 배치 캐시 무효화 등에서 Pipeline으로 10배 속도 향상

In [ ]:
# 실험 1: Redis 단건 vs Pipeline

r = aioredis.from_url("redis://localhost:6379", decode_responses=True)
N = 1000  # 저장할 키 수

# ─── 방법 1: 단건 SET (N번 왕복) ───
print(f"🔴 Redis 단건 SET ({N}개)...")
await r.delete(*[f"bench:single:{i}" for i in range(N)])

start = time.perf_counter()
for i in range(N):
    await r.set(f"bench:single:{i}", f"value_{i}", ex=60)
single_ms = (time.perf_counter() - start) * 1000
single_throughput = N / (single_ms / 1000)

print(f"  소요: {single_ms:.1f}ms | 처리량: {single_throughput:,.0f} ops/s")

# ─── 방법 2: Pipeline (1번 왕복) ───
print(f"\n🟢 Redis Pipeline ({N}개)...")
await r.delete(*[f"bench:pipe:{i}" for i in range(N)])

start = time.perf_counter()
pipe = r.pipeline()
for i in range(N):
    pipe.set(f"bench:pipe:{i}", f"value_{i}", ex=60)
await pipe.execute()
pipe_ms = (time.perf_counter() - start) * 1000
pipe_throughput = N / (pipe_ms / 1000)

print(f"  소요: {pipe_ms:.1f}ms | 처리량: {pipe_throughput:,.0f} ops/s")

# ─── 결과 비교 ───
speedup = single_ms / pipe_ms
print(f"\n📊 결과:")
print(f"  단건:     {single_ms:8.1f}ms  ({single_throughput:8,.0f} ops/s)")
print(f"  Pipeline: {pipe_ms:8.1f}ms  ({pipe_throughput:8,.0f} ops/s)")
print(f"  ⚡ Pipeline이 {speedup:.1f}배 빠름")

# ─── 결과 저장 ───
redis_results = {
    "single": {"ms": single_ms, "throughput": single_throughput},
    "pipeline": {"ms": pipe_ms, "throughput": pipe_throughput},
    "speedup": speedup
}

await r.aclose()

---

## 실험 2: Kafka — 기본 vs 배치 처리량 비교

### Kafka Producer 핵심 튜닝 파라미터

```text
[기본 설정]                        [튜닝 설정]

batch_size = 16,384 B (16KB)      batch_size = 65,536 B (64KB)
linger_ms  = 0 (즉시 전송)         linger_ms  = 10ms (배치 충전 대기)
compression = none                 compression = lz4

→ 메시지마다 즉시 전송              → 10ms 동안 쌓아서 한 번에 전송
→ 네트워크 요청 수 많음             → 네트워크 요청 수 줄어듦
→ 레이턴시 낮음                    → 처리량 높음 (레이턴시 소폭 상승)
```

**언제 어떤 설정을?**
- `linger_ms=0`: 실시간 알림, 낮은 레이턴시 요구
- `linger_ms=10~50`: 대용량 데이터 수집, 로그 파이프라인

In [ ]:
# 실험 2: Kafka 기본 vs Batch 브로커 랩 API 비교
# (브로커 랩은 두 엔드포인트를 별도로 제공합니다)

print("🔵 Kafka 처리량 비교 실험")
print("="*50)

counts = [100, 500, 1000]
kafka_results = {"basic": [], "batch": []}

for count in counts:
    print(f"\n[{count}개 메시지]")
    
    # Basic (개별 send_and_wait — 높은 내구성, 낮은 처리량)
    try:
        result = await api("post", "/benchmark/kafka", json={"message_count": count})
        throughput = result.get("throughput_msg_per_sec", 0)
        avg_lat = result.get("avg_latency_ms", 0)
        kafka_results["basic"].append(throughput)
        print(f"  Basic:  {throughput:8,.0f} msg/s  avg_lat={avg_lat:.2f}ms")
    except Exception as e:
        kafka_results["basic"].append(0)
        print(f"  Basic:  오류 — {e}")
    
    await asyncio.sleep(1)  # 락 해제 대기
    
    # Batch (send + flush — 높은 처리량, 배치 단위 내구성)
    try:
        result = await api("post", "/benchmark/kafka-batch", json={"message_count": count})
        throughput = result.get("throughput_msg_per_sec", 0)
        avg_lat = result.get("avg_latency_ms", 0)
        kafka_results["batch"].append(throughput)
        print(f"  Batch:  {throughput:8,.0f} msg/s  avg_lat={avg_lat:.2f}ms")
    except Exception as e:
        kafka_results["batch"].append(0)
        print(f"  Batch:  오류 — {e}")
    
    await asyncio.sleep(1)

print("\n📌 Kafka Basic vs Batch 차이:")
print("  Basic:  매 메시지마다 send_and_wait → 높은 내구성")
print("  Batch:  send() 모두 후 flush() → 높은 처리량")
print("  실무: 로그 파이프라인은 Batch, 결제/주문은 Basic")

---

## 실험 3: 전체 브로커 처리량 비교 + 시각화

In [ ]:
# 실험 3: 전체 브로커 종합 벤치마크

print("🏁 전체 브로커 종합 벤치마크 (1,000개 메시지)")
print("="*55)
print("(약 30~60초 소요)")

result = await api("post", "/benchmark/all", json={"message_count": 1000})
broker_data = result.get("benchmark_results", {})

print("\n📊 결과:")
print(f"{'브로커':<20} {'처리량(msg/s)':>15} {'평균지연(ms)':>14} {'P99(ms)':>10}")
print("-"*62)

chart_data = []
for name, data in broker_data.items():
    if "error" in data:
        print(f"{name:<20} {'오류':>15}")
        continue
    throughput = data.get("throughput_msg_per_sec", 0)
    avg_lat = data.get("avg_latency_ms", 0)
    p99 = data.get("p99_latency_ms", 0)
    print(f"{name:<20} {throughput:>15,.0f} {avg_lat:>14.3f} {p99:>10.3f}")
    chart_data.append({"name": name, "throughput": throughput, "avg_lat": avg_lat, "p99": p99})

# 정렬
chart_data.sort(key=lambda x: x["throughput"], reverse=True)

print("\n🏆 처리량 순위:")
for i, d in enumerate(chart_data, 1):
    bar = "█" * int(d['throughput'] / max(x['throughput'] for x in chart_data) * 30)
    print(f"  {i}위 {d['name']:<18} {bar:<30} {d['throughput']:>8,.0f} msg/s")

In [ ]:
# 시각화 (matplotlib 설치 시)

if PLOT_AVAILABLE and chart_data:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Message Broker Performance Comparison (1,000 messages)', 
                 fontsize=14, fontweight='bold', y=1.02)

    colors = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71', '#3498db']
    names = [d['name'] for d in chart_data]

    # 처리량
    ax1 = axes[0]
    bars = ax1.barh(names, [d['throughput'] for d in chart_data], color=colors[:len(chart_data)])
    ax1.set_xlabel('Throughput (msg/s)')
    ax1.set_title('Throughput (높을수록 좋음)')
    for bar, d in zip(bars, chart_data):
        ax1.text(bar.get_width() * 0.5, bar.get_y() + bar.get_height()/2,
                 f"{d['throughput']:,.0f}", ha='center', va='center',
                 color='white', fontweight='bold', fontsize=9)
    ax1.invert_yaxis()

    # 평균 레이턴시
    ax2 = axes[1]
    bars2 = ax2.barh(names, [d['avg_lat'] for d in chart_data], color=colors[:len(chart_data)])
    ax2.set_xlabel('Avg Latency (ms)')
    ax2.set_title('Avg Latency (낮을수록 좋음)')
    for bar, d in zip(bars2, chart_data):
        ax2.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2,
                 f"{d['avg_lat']:.2f}ms", va='center', fontsize=9)
    ax2.invert_yaxis()

    # P99 레이턴시
    ax3 = axes[2]
    bars3 = ax3.barh(names, [d['p99'] for d in chart_data], color=colors[:len(chart_data)])
    ax3.set_xlabel('P99 Latency (ms)')
    ax3.set_title('P99 Latency (Tail Latency)')
    for bar, d in zip(bars3, chart_data):
        ax3.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2,
                 f"{d['p99']:.2f}ms", va='center', fontsize=9)
    ax3.invert_yaxis()

    plt.tight_layout()
    plt.savefig('broker_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 차트 저장: broker_performance.png")
else:
    print("(matplotlib 없음 — 수치 결과만 표시됨)")

---

## 실험 4: 메시지 수 증가에 따른 처리량 선형성 확인

좋은 메시지 브로커는 메시지 수가 증가해도 **처리량이 선형으로 유지**되어야 합니다.  
떨어지면 → 병목(backpressure) 발생 신호

In [ ]:
# 실험 4: 메시지 수 스케일 테스트 (Redis Pub/Sub 기준)

print("📈 Redis Pub/Sub — 메시지 수별 처리량 선형성 테스트")
print("="*55)

scale_counts = [100, 500, 1000, 5000]
scale_results = []

for count in scale_counts:
    try:
        result = await api("post", "/benchmark/redis", json={"message_count": count})
        throughput = result.get("throughput_msg_per_sec", 0)
        avg_lat = result.get("avg_latency_ms", 0)
        total_ms = result.get("total_ms", 0)
        scale_results.append({"count": count, "throughput": throughput, "avg_lat": avg_lat})
        print(f"  {count:>5}개: {throughput:>10,.0f} msg/s  avg={avg_lat:.3f}ms  total={total_ms:.0f}ms")
    except Exception as e:
        scale_results.append({"count": count, "throughput": 0, "avg_lat": 0})
        print(f"  {count:>5}개: 오류 — {e}")
    
    await asyncio.sleep(0.5)

# 선형성 분석
valid = [r for r in scale_results if r["throughput"] > 0]
if len(valid) >= 2:
    first_tp = valid[0]["throughput"]
    last_tp = valid[-1]["throughput"]
    ratio = last_tp / first_tp
    
    print(f"\n📊 선형성 분석:")
    print(f"  {valid[0]['count']}개 처리량:  {first_tp:,.0f} msg/s")
    print(f"  {valid[-1]['count']}개 처리량: {last_tp:,.0f} msg/s")
    print(f"  비율: {ratio:.2f}x")
    
    if ratio > 0.8:
        print(f"  ✅ 처리량이 안정적으로 유지됨 (선형성 양호)")
    else:
        print(f"  ⚠️  메시지 수 증가에 따라 처리량이 감소 (병목 확인 필요)")

if PLOT_AVAILABLE and len(valid) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    counts_plot = [r['count'] for r in valid]
    throughputs = [r['throughput'] for r in valid]
    latencies = [r['avg_lat'] for r in valid]
    
    ax1.plot(counts_plot, throughputs, 'b-o', linewidth=2, markersize=8)
    ax1.fill_between(counts_plot, throughputs, alpha=0.1)
    ax1.set_xlabel('Message Count')
    ax1.set_ylabel('Throughput (msg/s)')
    ax1.set_title('Redis Pub/Sub 처리량 선형성')
    ax1.set_xscale('log')
    for x, y in zip(counts_plot, throughputs):
        ax1.annotate(f'{y:,.0f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center')
    
    ax2.plot(counts_plot, latencies, 'r-o', linewidth=2, markersize=8)
    ax2.set_xlabel('Message Count')
    ax2.set_ylabel('Avg Latency (ms)')
    ax2.set_title('메시지 수에 따른 레이턴시 변화')
    ax2.set_xscale('log')
    
    plt.tight_layout()
    plt.savefig('redis_scalability.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## 실험 5: P99 Tail Latency — 왜 평균만 보면 안 되나?

```text
[평균의 함정]

99개의 요청이 0.1ms → 1개의 요청이 100ms
→ 평균: 1.09ms (괜찮아 보임)
→ P99: 100ms  (1%의 사용자가 100ms 대기!)

프로덕션에서는 P99, P99.9를 기준으로 SLA를 정합니다.
```

**beanllm과의 연관:**  
beanllm 내부의 `BenchmarkResult` 클래스가 `p50`, `p99`를 추적하는 이유!  
→ 브로커 랩의 `BenchmarkResult`에 이미 추가되어 있습니다.

In [ ]:
# 실험 5: Tail Latency 분석
import redis.asyncio as aioredis

print("📐 P50 / P95 / P99 Tail Latency 직접 측정")
print("(Redis 개별 SET 1000회 측정)")
print("="*50)

r = aioredis.from_url("redis://localhost:6379", decode_responses=True)

latencies_ms = []
N = 1000

for i in range(N):
    t0 = time.perf_counter()
    await r.set(f"latency_test:{i}", f"v_{i}", ex=10)
    latencies_ms.append((time.perf_counter() - t0) * 1000)

latencies_ms.sort()

p50  = latencies_ms[int(N * 0.50)]
p90  = latencies_ms[int(N * 0.90)]
p95  = latencies_ms[int(N * 0.95)]
p99  = latencies_ms[int(N * 0.99)]
p999 = latencies_ms[int(N * 0.999)] if N >= 1000 else latencies_ms[-1]
avg  = statistics.mean(latencies_ms)
max_ = max(latencies_ms)

print(f"\n  평균(avg):    {avg:.3f}ms")
print(f"  P50 중앙값:   {p50:.3f}ms")
print(f"  P90:          {p90:.3f}ms")
print(f"  P95:          {p95:.3f}ms")
print(f"  P99:          {p99:.3f}ms  ← SLA 기준")
print(f"  P99.9:        {p999:.3f}ms")
print(f"  최대(max):    {max_:.3f}ms")

print(f"\n  P99/평균 비율: {p99/avg:.1f}x")
print(f"  → 상위 1% 요청이 평균보다 {p99/avg:.1f}배 느림")

print(f"\n📌 실무 SLA 예시:")
print(f"  Redis Cache:   P99 < 10ms  → 현재: {p99:.1f}ms {'✅' if p99 < 10 else '⚠️'}")
print(f"  API Response:  P99 < 100ms → 현재: {p99:.1f}ms {'✅' if p99 < 100 else '⚠️'}")

if PLOT_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # 히스토그램
    ax1 = axes[0]
    ax1.hist(latencies_ms, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax1.axvline(avg, color='orange', linestyle='--', label=f'Mean: {avg:.2f}ms')
    ax1.axvline(p99, color='red', linestyle='--', label=f'P99: {p99:.2f}ms')
    ax1.set_xlabel('Latency (ms)')
    ax1.set_ylabel('Count')
    ax1.set_title(f'Redis SET Latency Distribution (N={N})')
    ax1.legend()
    
    # 백분위수
    ax2 = axes[1]
    percentiles = [50, 75, 90, 95, 99, 99.9]
    values = [latencies_ms[min(int(N*p/100), N-1)] for p in percentiles]
    ax2.bar([f'P{p}' for p in percentiles], values, color=['#2ecc71','#3498db','#9b59b6','#e67e22','#e74c3c','#c0392b'])
    ax2.set_xlabel('Percentile')
    ax2.set_ylabel('Latency (ms)')
    ax2.set_title('Percentile Latency (Tail Latency)')
    for i, v in enumerate(values):
        ax2.text(i, v + max(values)*0.01, f'{v:.2f}ms', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('tail_latency.png', dpi=150, bbox_inches='tight')
    plt.show()

await r.aclose()

---

## 📋 핵심 정리: 프로덕션 튜닝 치트시트

### Kafka 튜닝

```python
# 고처리량 파이프라인 (로그, 이벤트 수집)
AIOKafkaProducer(
    batch_size=65536,       # 64KB (기본 16KB → 4x 배치 효율)
    linger_ms=10,           # 10ms 대기 → 배치 충전 시간
    compression_type="lz4", # CPU 효율 최고의 압축
    acks="all",             # 안전성 보장 (처리량 약간 감소)
)

# 저레이턴시 (실시간 결제, 알림)
AIOKafkaProducer(
    batch_size=16384,       # 기본값 유지
    linger_ms=0,            # 즉시 전송
    compression_type="none",# 압축 없음 → CPU 절약
    acks="all",
)
```

### Redis 튜닝

```python
# 대량 쓰기: 반드시 Pipeline 사용!
pipe = r.pipeline()
for key, value in data.items():
    pipe.set(key, value, ex=3600)
await pipe.execute()  # → 단건 대비 5~20x 빠름

# 읽기 최적화: mget (여러 키 한번에)
values = await r.mget(["key1", "key2", "key3"])  # GET 3번 대신 1번
```

### 모니터링 지표 체크리스트

| 지표 | 정상 | 주의 | 위험 |
|------|------|------|------|
| Redis P99 레이턴시 | < 1ms | 1~10ms | > 10ms |
| Kafka Consumer Lag | 0~100 | 1K~10K | > 100K |
| RabbitMQ Queue Depth | 0~1K | 1K~10K | > 10K |
| 처리량 대비 목표 | > 90% | 70~90% | < 70% |

```promql
# Prometheus로 실시간 모니터링
# P99 레이턴시
histogram_quantile(0.99, broker_publish_latency_seconds_bucket)

# 초당 처리량
rate(broker_publish_total[1m])
```

## 다음 단계

- [← 19. beanllm × Broker 시너지](./19_beanllm_broker_synergy.ipynb)
- [← 18. AI 시맨틱 캐시](./18_challenge_ai_semantic_cache.ipynb)
- [📊 Grafana 대시보드](http://localhost:3000) — 실시간 메트릭 시각화
- [📈 Prometheus](http://localhost:9090) — 직접 PromQL 쿼리